# 1. Apriori
### Limpieza de los datos
Para la implementacion de nuestro algoritmo de Apriori, despues de cargar el data set y realizar una pequeña revision, optamos por conservar los datos que aparecian con la etiqueta de "CONFIDENCIAL" ya que el eliminarlas podria afectar el analisis que estamos por hacer.

Apriori no trabaja con numeros continuos por lo tanto se realizo la discretizacion de las fechas junto con el calculo de la `EDAD` para poder trabajar en categorias, finalmente definimos las columnas con las que vamos a trabajar y terminamos la preparacion para el analisis











In [1]:
pip install --upgrade jupyter_client

In [10]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("data_secretariado.csv")

# 2. Calcular y discretizar la EDAD
df['FECHA_NAC_DT'] = pd.to_datetime(df['FECHA_NACIMIENTO'], errors='coerce')
df['FECHA_DESAP_DT'] = pd.to_datetime(df['FECHA_DESAPARICION'], errors='coerce')
df['EDAD_CALCULADA'] = df['FECHA_DESAP_DT'].dt.year - df['FECHA_NAC_DT'].dt.year

# Discretizamos la edad en grupos (cubetas)
bins_edad = [0, 11, 17, 29, 59, 120]
labels_edad = ['0-11', '12-17', '18-29', '30-59', '60+']
df['GRUPO_EDAD'] = pd.cut(df['EDAD_CALCULADA'], bins=bins_edad, labels=labels_edad)

# 3. Agrupamos el MES de DESAPARICION en 4 grupos (cubetas)
df['MES_NUMERO'] = df['FECHA_DESAP_DT'].dt.month

bins_mes = [0, 3, 6, 9, 12]
labels_mes = ['Trimestre 1 (Ene-Mar)', 'Trimestre 2 (Abr-Jun)', 'Trimestre 3 (Jul-Sep)', 'Trimestre 4 (Oct-Dic)']
df['PERIODO_DESAPARICION'] = pd.cut(df['MES_NUMERO'], bins=bins_mes, labels=labels_mes)

# 4. Restauramos los valores CONFIDENCIAL
df['GRUPO_EDAD'] = df['GRUPO_EDAD'].astype(str).replace('nan', 'CONFIDENCIAL')
df['PERIODO_DESAPARICION'] = df['PERIODO_DESAPARICION'].astype(str).replace('nan', 'CONFIDENCIAL')

# 5. Transformación Transaccional
columnas_reglas = ['SEXO', 'GRUPO_EDAD', 'ENTIDAD', 'PERIODO_DESAPARICION', 'ESTATUS_VICTIMA']

# 6. Creamos una lista de transacciones, excluyendo los NaN
transactions = []

for idx, row in df[columnas_reglas].iterrows():
    # Filtramos solo los valores que NO son NaN
    transaccion = [str(val) for val in row.values if pd.notna(val)]
    if transaccion:
        transactions.append(transaccion)

print(f"Total de transacciones creadas: {len(transactions)}")

Total de transacciones creadas: 133887


Aplicamos Apriori apoyandonos de `mlxtend`

In [11]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

te = TransactionEncoder()
te_array = te.fit(transactions).transform(transactions)

df_encoded = pd.DataFrame(te_array, columns=te.columns_)

frequent_itemsets = apriori(df_encoded, min_support=0.1, use_colnames=True)

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.6)

rules_filtradas = rules[(rules['confidence'] >= 0.6) & (rules['lift'] > 1)].sort_values(by='lift', ascending=False)

Finalmente imprimimos nuestras reglas de asociacion resultantes de haber filtrado mediando un `confidence` mayor o igual a 0.6 y un `lift` mayor a 1

In [12]:
from tabulate import tabulate

def format_rules(df):
    tabla = df.copy()

    # Convertir sets a texto
    tabla['antecedents'] = tabla['antecedents'].apply(lambda x: ', '.join(list(x)))
    tabla['consequents'] = tabla['consequents'].apply(lambda x: ', '.join(list(x)))

    # Redondear métricas
    tabla['support'] = tabla['support'].round(3)
    tabla['confidence'] = tabla['confidence'].round(3)
    tabla['lift'] = tabla['lift'].round(3)

    return tabla[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

tabla_final = format_rules(rules_filtradas)

print(tabulate(tabla_final.head(20), headers='keys', tablefmt='grid', showindex=False))

+-------------------------------------+----------------------+-----------+--------------+--------+
| antecedents                         | consequents          |   support |   confidence |   lift |
+=====================================+======================+===========+==============+========+
| 30-59                               | DESAPARECIDA, HOMBRE |     0.147 |        0.815 |  1.795 |
+-------------------------------------+----------------------+-----------+--------------+--------+
| 30-59                               | HOMBRE               |     0.156 |        0.866 |  1.793 |
+-------------------------------------+----------------------+-----------+--------------+--------+
| 30-59, DESAPARECIDA                 | HOMBRE               |     0.147 |        0.864 |  1.79  |
+-------------------------------------+----------------------+-----------+--------------+--------+
| 18-29                               | DESAPARECIDA, HOMBRE |     0.101 |        0.755 |  1.663 |
+---------